In [23]:
import os
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from scipy.stats import pearsonr, shapiro, kstest, anderson, jarque_bera, gaussian_kde
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE, mutual_info_regression
from sklearn.linear_model import ElasticNet

warnings.filterwarnings("ignore")

print("="*80)
print("模型加载与完整残差分析程序（仅推理，不训练）")
print("="*80)

# ====================== 路径配置 ======================
# 修改为你的实际路径
data_path = r"D:\文成数据库\Nb-Si数据库2.28-高温压缩.xlsx"
model_dir = r"D:\博一下\manuscript3-机器学习\nc\nc修改9.27\10.15\10.21\1.20-回复审稿人1-C1\HTC修复1.20-架构32_16-终极优化版1.21.8"
output_dir = os.path.join(model_dir, 'Complete_Residual_Analysis-2')
os.makedirs(output_dir, exist_ok=True)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# ====================== 数据加载 ======================
print("\n[1/4] 加载数据...")
df = pd.read_excel(data_path)

features_name = [
    "Nb", "Si", "Ti", "Al", "Cr", "Hf", "Zr", "Mo", "Fe", "B", "V", 
    "VEC", "x", "Δx", "ΔHmix", "ΔSmix", "ΔG", "Tm", "ΔTm", "ρ", "r", "am", "Δa", "δ", "Ω", "λ",
    "Mean_A1 atomic number", "Mean_A6 atomic weight", "Mean_E2 electronegativity (Pauling)",
    "Mean_E5 energy ionization first", "Mean_E13 Electron affinity", "Mean_Electrophilicity Index",
    "Mean_Speed of Sound", "Mean_Neutron Cross Section", "Mean_Neutron Mass Absorption",
    "Mean_Space Group Number", "Mean_Glawe Number", "Mean_C1 temperature melting",
    "Mean_C2 temperature boiling", "Mean_density", "Mean_C3 enthalpy vaporization",
    "Mean_C4 enthalpy melting", "Mean_C5 enthalpy atomization", "Mean_热导率W/(mk)",
    "Mean_电导率(MS/m)", "Mean_电阻率(mΩ)", "Mean_热膨胀(1/k)", "Mean_比热容J/g/k",
    "Mean_摩尔热容(J/mol/k)", "Mean_摩尔体积(cm3/mol)", "Mean_莫氏硬度(MPa)",
    "Mean_covalent radius Cordero (pm)", "Mean_covalent radius Pyykko(Single Bond) (pm)",
    "Mean_covalent radius Pyykko(Double Bond) (pm)", "Mean_Pyykko(Triple Bond) (pm)",
    "Mean_S10 Lattice Constants a", "Mean_S11 Lattice Constants b", "Mean_S12 Lattice Constants c",
    "Mean_S13 radii atomic (coordination number 12) (pm)", "Mean_metallic radius(pm)",
    "Mean_metallic radius 12 Neighbors(pm)",
    "Var_A1 atomic number", "Var_A6 atomic weight", "Var_E2 electronegativity (Pauling)",
    "Var_E5 energy ionization first", "Var_E13 Electron affinity", "Var_Electrophilicity Index",
    "Var_Speed of Sound", "Var_Neutron Cross Section", "Var_Neutron Mass Absorption",
    "Var_Space Group Number", "Var_Glawe Number", "Var_C1 temperature melting",
    "Var_C2 temperature boiling", "Var_density", "Var_C3 enthalpy vaporization",
    "Var_C4 enthalpy melting", "Var_C5 enthalpy atomization", "Var_热导率W/(mk)",
    "Var_电导率(MS/m)", "Var_电阻率(mΩ)", "Var_热膨胀(1/k)", "Var_比热容J/g/k",
    "Var_摩尔热容(J/mol/k)", "Var_摩尔体积(cm3/mol)", "Var_莫氏硬度(MPa)",
    "Var_covalent radius Cordero (pm)", "Var_covalent radius Pyykko(Single Bond) (pm)",
    "Var_covalent radius Pyykko(Double Bond) (pm)", "Var_Pyykko(Triple Bond) (pm)",
    "Var_S10 Lattice Constants a", "Var_S11 Lattice Constants b", "Var_S12 Lattice Constants c",
    "Var_S13 radii atomic (coordination number 12) (pm)", "Var_metallic radius(pm)",
    "Var_metallic radius 12 Neighbors(pm)",
]

target_col = 'high temperature compression'
X_all = df[features_name].values
y_all = df[target_col].values
y_all_np = np.array(y_all, dtype=float)

# ====================== 特征工程（复现训练时的流程）======================
print("\n[2/4] 特征工程...")
scaler_for_selection = StandardScaler()
X_all_scaled = scaler_for_selection.fit_transform(X_all)
n_samples, n_features = X_all_scaled.shape

# PCC去除冗余
pcc_matrix = np.zeros((n_features, n_features))
for i in range(n_features):
    for j in range(i + 1, n_features):
        pcc_val, _ = pearsonr(X_all_scaled[:, i], X_all_scaled[:, j])
        pcc_matrix[i, j] = pcc_val
        pcc_matrix[j, i] = pcc_val

pcc_threshold = 0.95
redundant_features = set()
for i in range(n_features):
    for j in range(i + 1, n_features):
        if abs(pcc_matrix[i, j]) > pcc_threshold:
            pcc_i, _ = pearsonr(X_all_scaled[:, i], y_all_np)
            pcc_j, _ = pearsonr(X_all_scaled[:, j], y_all_np)
            if abs(pcc_i) < abs(pcc_j):
                redundant_features.add(i)
            else:
                redundant_features.add(j)

non_redundant_indices = [i for i in range(n_features) if i not in redundant_features]
X_nonredundant = X_all_scaled[:, non_redundant_indices]
nr_features_name = [features_name[i] for i in non_redundant_indices]

# 过滤法
corr_threshold = 0.09
pcc_with_target = []
for i in range(X_nonredundant.shape[1]):
    p_val, _ = pearsonr(X_nonredundant[:, i], y_all_np)
    pcc_with_target.append(abs(p_val))

pcc_indices = [i for i, v in enumerate(pcc_with_target) if v > corr_threshold]
mic_values = mutual_info_regression(X_nonredundant, y_all_np)
mic_threshold = 0.08
mic_indices = [i for i, v in enumerate(mic_values) if v > mic_threshold]
selected_indices_filter = sorted(list(set(pcc_indices).intersection(set(mic_indices))))
filtered_features = [nr_features_name[i] for i in selected_indices_filter]
X_filter = X_nonredundant[:, selected_indices_filter]

# RFE
n_features_to_select = 6
elasticnet_estimator = ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=10000, random_state=0)
selector = RFE(estimator=elasticnet_estimator, n_features_to_select=n_features_to_select)
selector = selector.fit(X_filter, y_all_np)
selected_mask_rfe = selector.support_
final_features_rfe = [filtered_features[i] for i, sel in enumerate(selected_mask_rfe) if sel]

print(f"  最终特征: {final_features_rfe}")

X_final_original = df[final_features_rfe].values
final_scaler = StandardScaler()
X_final_scaled = final_scaler.fit_transform(X_final_original)

# ====================== 模型定义 ======================
class OptimalModel(nn.Module):
    def __init__(self, input_dim=6):
        super(OptimalModel, self).__init__()
        self.layer1 = nn.Linear(input_dim, 32)
        self.bn1 = nn.BatchNorm1d(32)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(0.05)
        self.layer2 = nn.Linear(32, 16)
        self.bn2 = nn.BatchNorm1d(16)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(0.02)
        self.layer3 = nn.Linear(16, 1)

    def forward(self, x):
        x = self.layer1(x)
        x = self.bn1(x)
        x = self.relu1(x)
        x = self.dropout1(x)
        x = self.layer2(x)
        x = self.bn2(x)
        x = self.relu2(x)
        x = self.dropout2(x)
        x = self.layer3(x)
        return x

# ====================== 加载已训练模型 ======================
print("\n[3/4] 加载已训练模型...")
model_path = os.path.join(model_dir, 'best_model.pth')

# 检查文件是否存在
if not os.path.exists(model_path):
    raise FileNotFoundError(f"模型文件不存在: {model_path}")

# 加载checkpoint（添加weights_only=False以兼容旧版本保存的模型）
checkpoint = torch.load(model_path, map_location=device, weights_only=False)

# 提取信息
saved_seed = checkpoint['random_seed']
saved_metrics = checkpoint['final_metrics']

print(f"  ✓ 模型文件: {model_path}")
print(f"  ✓ 随机种子: {saved_seed}")
print(f"  ✓ 保存的性能: Train R²={saved_metrics['train_r2']:.4f}, Test R²={saved_metrics['test_r2']:.4f}")

# 初始化模型并加载权重
model = OptimalModel(len(final_features_rfe)).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()  # 设置为评估模式
print(f"  ✓ 模型已成功加载并设置为评估模式")

# ====================== 使用保存的种子划分数据 ======================
print(f"\n  使用种子 {saved_seed} 划分数据...")
feature_rfe = pd.DataFrame(X_final_scaled, columns=final_features_rfe)
target = pd.Series(y_all_np, name=target_col)

x_train_np, x_test_np, y_train_np, y_test_np = train_test_split(
    feature_rfe.values, target.values, test_size=0.2, random_state=saved_seed
)

# 转换为张量
train_x = torch.from_numpy(x_train_np.astype(np.float32)).to(device)
test_x = torch.from_numpy(x_test_np.astype(np.float32)).to(device)

# ====================== 模型推理 ======================
print("\n  进行模型推理...")
with torch.no_grad():
    # 训练集预测
    train_pred = model(train_x).cpu().numpy().flatten()
    
    # 测试集预测
    test_pred = model(test_x).cpu().numpy().flatten()
    
    # 全数据集预测
    all_features_tensor = torch.from_numpy(X_final_scaled.astype(np.float32)).to(device)
    all_pred = model(all_features_tensor).cpu().numpy().flatten()

# 计算性能指标
def calculate_mape(y_true, y_pred):
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

train_r2 = r2_score(y_train_np, train_pred)
train_rmse = np.sqrt(mean_squared_error(y_train_np, train_pred))
train_mae = mean_absolute_error(y_train_np, train_pred)
train_mape = calculate_mape(y_train_np, train_pred)

test_r2 = r2_score(y_test_np, test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test_np, test_pred))
test_mae = mean_absolute_error(y_test_np, test_pred)
test_mape = calculate_mape(y_test_np, test_pred)

print(f"\n  验证模型性能:")
print(f"  训练集: R²={train_r2:.4f}, RMSE={train_rmse:.4f}, MAE={train_mae:.4f}")
print(f"  测试集: R²={test_r2:.4f}, RMSE={test_rmse:.4f}, MAE={test_mae:.4f}")

# ====================== 残差计算 ======================
print("\n[4/4] 完整残差分析...")
train_residuals = y_train_np - train_pred
test_residuals = y_test_np - test_pred
all_residuals = y_all_np - all_pred

train_residuals_std = (train_residuals - train_residuals.mean()) / train_residuals.std()
test_residuals_std = (test_residuals - test_residuals.mean()) / test_residuals.std()
all_residuals_std = (all_residuals - all_residuals.mean()) / all_residuals.std()

print(f"\n  残差统计:")
print(f"  训练集: Mean={train_residuals.mean():.4f}, Std={train_residuals.std():.4f}")
print(f"  测试集: Mean={test_residuals.mean():.4f}, Std={test_residuals.std():.4f}")
print(f"  全数据: Mean={all_residuals.mean():.4f}, Std={all_residuals.std():.4f}")

# ====================== 统计检验 ======================
shapiro_stat, shapiro_p = shapiro(all_residuals)
ks_stat, ks_p = kstest(all_residuals_std, 'norm')
anderson_result = anderson(all_residuals, dist='norm')
jb_stat, jb_p = jarque_bera(all_residuals)
dw_stat = durbin_watson(all_residuals)
X_with_const = np.column_stack([np.ones(len(all_pred)), all_pred])
bp_stat, bp_p, _, _ = het_breuschpagan(all_residuals, X_with_const)
corr_all, p_all = pearsonr(all_pred, all_residuals)

print(f"\n  统计检验:")
print(f"  Shapiro-Wilk: p={shapiro_p:.4f} {'✓' if shapiro_p > 0.05 else '⚠'}")
print(f"  K-S Test: p={ks_p:.4f} {'✓' if ks_p > 0.05 else '⚠'}")
print(f"  Jarque-Bera: p={jb_p:.4f} {'✓' if jb_p > 0.05 else '⚠'}")
print(f"  Durbin-Watson: {dw_stat:.4f} {'✓' if 1.5 < dw_stat < 2.5 else '⚠'}")
print(f"  Breusch-Pagan: p={bp_p:.4f} {'✓' if bp_p > 0.05 else '⚠'}")
print(f"  Pred vs Residual: r={corr_all:.4f} {'✓' if abs(corr_all) < 0.1 else ('✓(weak)' if abs(corr_all) < 0.3 else '⚠')}")

passed_tests = sum([
    shapiro_p > 0.05, ks_p > 0.05, jb_p > 0.05,
    anderson_result.statistic < anderson_result.critical_values[2],
    1.5 < dw_stat < 2.5, bp_p > 0.05,
    (abs(corr_all) < 0.1 and p_all > 0.05) or (abs(corr_all) < 0.3) * 0.5
])
print(f"  通过检验: {passed_tests:.1f}/7 ({passed_tests/7*100:.1f}%)")

# ====================== 生成所有可视化数据 ======================
print("\n  生成所有可视化数据...")

# 1. 实际值 vs 预测值（散点图数据）
scatter_data = pd.DataFrame({
    'Actual_HTC': np.concatenate([y_train_np, y_test_np]),
    'Predicted_HTC': np.concatenate([train_pred, test_pred]),
    'Dataset': ['Train']*len(y_train_np) + ['Test']*len(y_test_np)
})

# 2. 残差图数据
residual_plot_data = pd.DataFrame({
    'Predicted_HTC': np.concatenate([train_pred, test_pred]),
    'Residuals': np.concatenate([train_residuals, test_residuals]),
    'Dataset': ['Train']*len(y_train_np) + ['Test']*len(y_test_np)
})

# 3. 残差分布直方图数据
hist_data_train, hist_bins_train = np.histogram(train_residuals, bins=12, density=True)
hist_data_test, hist_bins_test = np.histogram(test_residuals, bins=8, density=True)
hist_data_all, hist_bins_all = np.histogram(all_residuals, bins=15, density=True)

hist_df_train = pd.DataFrame({
    'Bin_Center': (hist_bins_train[:-1] + hist_bins_train[1:]) / 2,
    'Density': hist_data_train,
    'Bin_Left': hist_bins_train[:-1],
    'Bin_Right': hist_bins_train[1:]
})

hist_df_test = pd.DataFrame({
    'Bin_Center': (hist_bins_test[:-1] + hist_bins_test[1:]) / 2,
    'Density': hist_data_test,
    'Bin_Left': hist_bins_test[:-1],
    'Bin_Right': hist_bins_test[1:]
})

hist_df_all = pd.DataFrame({
    'Bin_Center': (hist_bins_all[:-1] + hist_bins_all[1:]) / 2,
    'Density': hist_data_all,
    'Bin_Left': hist_bins_all[:-1],
    'Bin_Right': hist_bins_all[1:]
})

# 4. 正态分布拟合曲线
mu_train, sigma_train = train_residuals.mean(), train_residuals.std()
x_norm_train = np.linspace(train_residuals.min(), train_residuals.max(), 100)
y_norm_train = stats.norm.pdf(x_norm_train, mu_train, sigma_train)
normal_fit_df_train = pd.DataFrame({'X': x_norm_train, 'Normal_PDF': y_norm_train})

mu_test, sigma_test = test_residuals.mean(), test_residuals.std()
x_norm_test = np.linspace(test_residuals.min(), test_residuals.max(), 100)
y_norm_test = stats.norm.pdf(x_norm_test, mu_test, sigma_test)
normal_fit_df_test = pd.DataFrame({'X': x_norm_test, 'Normal_PDF': y_norm_test})

mu_all, sigma_all = all_residuals.mean(), all_residuals.std()
x_norm_all = np.linspace(all_residuals.min(), all_residuals.max(), 100)
y_norm_all = stats.norm.pdf(x_norm_all, mu_all, sigma_all)
normal_fit_df_all = pd.DataFrame({'X': x_norm_all, 'Normal_PDF': y_norm_all})

# 5. KDE曲线数据
kde_train = gaussian_kde(train_residuals)
x_range_train = np.linspace(train_residuals.min(), train_residuals.max(), 100)
kde_train_df = pd.DataFrame({
    'X': x_range_train,
    'KDE_Density': kde_train(x_range_train)
})

kde_test = gaussian_kde(test_residuals)
x_range_test = np.linspace(test_residuals.min(), test_residuals.max(), 100)
kde_test_df = pd.DataFrame({
    'X': x_range_test,
    'KDE_Density': kde_test(x_range_test)
})

# 6. Q-Q图数据
from scipy import stats as sp_stats

qq_data_train = sp_stats.probplot(train_residuals_std, dist="norm")
qq_df_train = pd.DataFrame({
    'Theoretical_Quantiles': qq_data_train[0][0],
    'Sample_Quantiles': qq_data_train[0][1],
    'Fit_Line_X': qq_data_train[0][0],
    'Fit_Line_Y': qq_data_train[0][0] * qq_data_train[1][0] + qq_data_train[1][1]
})

qq_data_test = sp_stats.probplot(test_residuals_std, dist="norm")
qq_df_test = pd.DataFrame({
    'Theoretical_Quantiles': qq_data_test[0][0],
    'Sample_Quantiles': qq_data_test[0][1],
    'Fit_Line_X': qq_data_test[0][0],
    'Fit_Line_Y': qq_data_test[0][0] * qq_data_test[1][0] + qq_data_test[1][1]
})

qq_data_all = sp_stats.probplot(all_residuals_std, dist="norm")
qq_df_all = pd.DataFrame({
    'Theoretical_Quantiles': qq_data_all[0][0],
    'Sample_Quantiles': qq_data_all[0][1],
    'Fit_Line_X': qq_data_all[0][0],
    'Fit_Line_Y': qq_data_all[0][0] * qq_data_all[1][0] + qq_data_all[1][1]
})

# 7. 箱线图统计数据
box_stats_df = pd.DataFrame({
    'Dataset': ['Train', 'Test', 'All'],
    'Min': [train_residuals.min(), test_residuals.min(), all_residuals.min()],
    'Q1': [np.percentile(train_residuals, 25), np.percentile(test_residuals, 25), np.percentile(all_residuals, 25)],
    'Median': [np.median(train_residuals), np.median(test_residuals), np.median(all_residuals)],
    'Q3': [np.percentile(train_residuals, 75), np.percentile(test_residuals, 75), np.percentile(all_residuals, 75)],
    'Max': [train_residuals.max(), test_residuals.max(), all_residuals.max()],
    'Mean': [train_residuals.mean(), test_residuals.mean(), all_residuals.mean()],
    'Std': [train_residuals.std(), test_residuals.std(), all_residuals.std()],
    'IQR': [np.percentile(train_residuals, 75) - np.percentile(train_residuals, 25),
            np.percentile(test_residuals, 75) - np.percentile(test_residuals, 25),
            np.percentile(all_residuals, 75) - np.percentile(all_residuals, 25)]
})

# 8. 标准化残差序列
std_residuals_df = pd.DataFrame({
    'Sample_Index': range(len(all_residuals_std)),
    'Standardized_Residuals': all_residuals_std,
    'Plus_2_Sigma': 2,
    'Minus_2_Sigma': -2,
    'Plus_3_Sigma': 3,
    'Minus_3_Sigma': -3
})

# 9. 异方差性检查（|残差| vs 预测值）
heteroscedasticity_df = pd.DataFrame({
    'Predicted_HTC': all_pred,
    'Abs_Residuals': np.abs(all_residuals)
})

# 趋势线
z = np.polyfit(all_pred, np.abs(all_residuals), 1)
trend_line_x = np.sort(all_pred)
trend_line_y = z[0] * trend_line_x + z[1]
trend_line_df = pd.DataFrame({
    'Predicted_HTC': trend_line_x,
    'Trend_Line': trend_line_y,
    'Slope': z[0],
    'Intercept': z[1]
})

# 10. ACF数据
max_lags_acf = min(20, len(all_residuals) // 2 - 1)
if max_lags_acf > 0:
    from statsmodels.tsa.stattools import acf
    acf_values = acf(all_residuals, nlags=max_lags_acf, fft=False)
    acf_df = pd.DataFrame({
        'Lag': range(len(acf_values)),
        'ACF': acf_values
    })
else:
    acf_df = pd.DataFrame({'Note': ['Sample size too small for ACF calculation']})

# 11. PACF数据
max_lags_pacf = min(15, len(all_residuals) // 2 - 2)
if max_lags_pacf > 0:
    try:
        from statsmodels.tsa.stattools import pacf
        pacf_values = pacf(all_residuals, nlags=max_lags_pacf)
        pacf_df = pd.DataFrame({
            'Lag': range(len(pacf_values)),
            'PACF': pacf_values
        })
    except:
        pacf_df = pd.DataFrame({'Note': ['PACF calculation failed']})
else:
    pacf_df = pd.DataFrame({'Note': ['Sample size too small for PACF calculation']})

# 12. 移动平均数据
window = min(3, len(all_residuals) // 3)
if window > 0:
    rolling_mean = pd.Series(all_residuals).rolling(window=window).mean()
    rolling_std = pd.Series(all_residuals).rolling(window=window).std()
    moving_avg_df = pd.DataFrame({
        'Sample_Index': range(len(all_residuals)),
        'Residuals': all_residuals,
        'Rolling_Mean': rolling_mean,
        'Rolling_Std': rolling_std,
        'Upper_Band': rolling_mean + rolling_std,
        'Lower_Band': rolling_mean - rolling_std
    })
else:
    moving_avg_df = pd.DataFrame({'Note': ['Sample size too small for moving average calculation']})

# 13. CUSUM数据
cumsum_residuals = np.cumsum(all_residuals)
cusum_df = pd.DataFrame({
    'Sample_Index': range(len(cumsum_residuals)),
    'CUSUM': cumsum_residuals,
    'Zero_Line': 0
})

# 14. 详细残差数据（带特征值）
detailed_residuals_df = pd.DataFrame()
for idx, feature in enumerate(final_features_rfe):
    detailed_residuals_df[feature] = final_scaler.inverse_transform(X_final_scaled)[:, idx]

detailed_residuals_df['Actual_HTC'] = y_all_np
detailed_residuals_df['Predicted_HTC'] = all_pred
detailed_residuals_df['Residual'] = all_residuals
detailed_residuals_df['Abs_Residual'] = np.abs(all_residuals)
detailed_residuals_df['Standardized_Residual'] = all_residuals_std
detailed_residuals_df['Squared_Residual'] = all_residuals**2
detailed_residuals_df['Percentage_Error'] = (all_residuals / y_all_np) * 100
detailed_residuals_df['APE (%)'] = np.abs((y_all_np - all_pred) / y_all_np) * 100

# 标记数据集类型
detailed_residuals_df['Dataset'] = 'Train'
for idx in range(len(y_all_np)):
    if any(np.isclose(y_all_np[idx], y_test_np) & np.isclose(all_pred[idx], test_pred, atol=0.01)):
        detailed_residuals_df.loc[idx, 'Dataset'] = 'Test'

# 15. 统计检验结果
stats_tests_df = pd.DataFrame({
    'Test_Name': ['Shapiro-Wilk (Normality)', 'Kolmogorov-Smirnov (Normality)', 'Anderson-Darling (Normality)',
                  'Jarque-Bera (Normality)', 'Durbin-Watson (Independence)', 'Breusch-Pagan (Homoscedasticity)',
                  'Correlation (Pred vs Residual)'],
    'Statistic': [shapiro_stat, ks_stat, anderson_result.statistic, jb_stat, dw_stat, bp_stat, corr_all],
    'P_Value': [shapiro_p, ks_p, np.nan, jb_p, np.nan, bp_p, p_all],
    'Critical_Value': [0.05, 0.05, anderson_result.critical_values[2], 0.05, np.nan, 0.05, 0.05],
    'Interpretation': [
        'Pass' if shapiro_p > 0.05 else 'Attention',
        'Pass' if ks_p > 0.05 else 'Attention',
        'Pass' if anderson_result.statistic < anderson_result.critical_values[2] else 'Attention',
        'Pass' if jb_p > 0.05 else 'Attention',
        'Pass' if 1.5 < dw_stat < 2.5 else 'Attention',
        'Pass' if bp_p > 0.05 else 'Attention',
        'Pass' if abs(corr_all) < 0.1 and p_all > 0.05 else ('Weak' if abs(corr_all) < 0.3 else 'Attention')
    ],
    'Threshold': ['p > 0.05', 'p > 0.05', f'< {anderson_result.critical_values[2]:.4f}', 'p > 0.05',
                  '1.5 < DW < 2.5', 'p > 0.05', '|r| < 0.1, p > 0.05']
})

# 16. 描述性统计
descriptive_stats_df = pd.DataFrame({
    'Metric': ['Count', 'Mean', 'Std Dev', 'Min', 'Max', 'Range', 'Skewness', 'Kurtosis', 
               'Q1', 'Median', 'Q3', 'IQR', '95% CI Lower', '95% CI Upper'],
    'Training_Set': [
        len(train_residuals), train_residuals.mean(), train_residuals.std(), 
        train_residuals.min(), train_residuals.max(), train_residuals.max() - train_residuals.min(),
        stats.skew(train_residuals), stats.kurtosis(train_residuals),
        np.percentile(train_residuals, 25), np.median(train_residuals), np.percentile(train_residuals, 75),
        np.percentile(train_residuals, 75) - np.percentile(train_residuals, 25),
        np.percentile(train_residuals, 2.5), np.percentile(train_residuals, 97.5)
    ],
    'Test_Set': [
        len(test_residuals), test_residuals.mean(), test_residuals.std(),
        test_residuals.min(), test_residuals.max(), test_residuals.max() - test_residuals.min(),
        stats.skew(test_residuals), stats.kurtosis(test_residuals),
        np.percentile(test_residuals, 25), np.median(test_residuals), np.percentile(test_residuals, 75),
        np.percentile(test_residuals, 75) - np.percentile(test_residuals, 25),
        np.percentile(test_residuals, 2.5), np.percentile(test_residuals, 97.5)
    ],
    'All_Data': [
        len(all_residuals), all_residuals.mean(), all_residuals.std(),
        all_residuals.min(), all_residuals.max(), all_residuals.max() - all_residuals.min(),
        stats.skew(all_residuals), stats.kurtosis(all_residuals),
        np.percentile(all_residuals, 25), np.median(all_residuals), np.percentile(all_residuals, 75),
        np.percentile(all_residuals, 75) - np.percentile(all_residuals, 25),
        np.percentile(all_residuals, 2.5), np.percentile(all_residuals, 97.5)
    ]
})

# 17. 模型性能汇总
performance_df = pd.DataFrame({
    'Dataset': ['Training', 'Test'],
    'R2': [train_r2, test_r2],
    'RMSE': [train_rmse, test_rmse],
    'MAE': [train_mae, test_mae],
    'MAPE (%)': [train_mape, test_mape],
    'Samples': [len(y_train_np), len(y_test_np)],
    'Features': [', '.join(final_features_rfe), ', '.join(final_features_rfe)]
})

# 18. 配置信息
config_df = pd.DataFrame({
    'Parameter': ['Random_Seed', 'Model_Architecture', 'Total_Parameters', 'Train_Samples', 
                  'Test_Samples', 'Total_Samples', 'Test_Size', 'Features'],
    'Value': [saved_seed, '6→32→16→1', '865', len(y_train_np), len(y_test_np), 
              len(y_all_np), '20%', ', '.join(final_features_rfe)]
})

# ====================== 保存所有数据到Excel ======================
print("\n  保存所有数据到Excel...")
excel_path = os.path.join(output_dir, 'Complete_Visualization_Data_For_Origin.xlsx')

with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    # 基础数据
    scatter_data.to_excel(writer, sheet_name='01_Scatter_Actual_vs_Pred', index=False)
    residual_plot_data.to_excel(writer, sheet_name='02_Residual_Plot', index=False)
    detailed_residuals_df.to_excel(writer, sheet_name='03_Detailed_Residuals', index=False)
    
    # 分布相关
    hist_df_train.to_excel(writer, sheet_name='04_Histogram_Train', index=False)
    hist_df_test.to_excel(writer, sheet_name='05_Histogram_Test', index=False)
    hist_df_all.to_excel(writer, sheet_name='06_Histogram_All', index=False)
    
    normal_fit_df_train.to_excel(writer, sheet_name='07_Normal_Fit_Train', index=False)
    normal_fit_df_test.to_excel(writer, sheet_name='08_Normal_Fit_Test', index=False)
    normal_fit_df_all.to_excel(writer, sheet_name='09_Normal_Fit_All', index=False)
    
    kde_train_df.to_excel(writer, sheet_name='10_KDE_Train', index=False)
    kde_test_df.to_excel(writer, sheet_name='11_KDE_Test', index=False)
    
    # Q-Q图
    qq_df_train.to_excel(writer, sheet_name='12_QQ_Plot_Train', index=False)
    qq_df_test.to_excel(writer, sheet_name='13_QQ_Plot_Test', index=False)
    qq_df_all.to_excel(writer, sheet_name='14_QQ_Plot_All', index=False)
    
    # 箱线图与标准化残差
    box_stats_df.to_excel(writer, sheet_name='15_Boxplot_Statistics', index=False)
    std_residuals_df.to_excel(writer, sheet_name='16_Standardized_Residuals', index=False)
    
    # 异方差性
    heteroscedasticity_df.to_excel(writer, sheet_name='17_Heteroscedasticity_Data', index=False)
    trend_line_df.to_excel(writer, sheet_name='18_Heteroscedasticity_Trend', index=False)
    
    # 时间序列分析
    acf_df.to_excel(writer, sheet_name='19_ACF_Data', index=False)
    pacf_df.to_excel(writer, sheet_name='20_PACF_Data', index=False)
    moving_avg_df.to_excel(writer, sheet_name='21_Moving_Average', index=False)
    cusum_df.to_excel(writer, sheet_name='22_CUSUM', index=False)
    
    # 统计与性能
    stats_tests_df.to_excel(writer, sheet_name='23_Statistical_Tests', index=False)
    descriptive_stats_df.to_excel(writer, sheet_name='24_Descriptive_Statistics', index=False)
    performance_df.to_excel(writer, sheet_name='25_Model_Performance', index=False)
    config_df.to_excel(writer, sheet_name='26_Configuration', index=False)

print(f"  ✓ Excel文件已保存: {excel_path}")

# ====================== 生成可视化图片（用于快速检查）======================
print("\n  生成可视化图片...")

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# 图1: 综合残差分析图（保持原有）
fig = plt.figure(figsize=(18, 12))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

# 训练集残差图
ax1 = fig.add_subplot(gs[0, 0])
ax1.scatter(train_pred, train_residuals, alpha=0.6, s=60, edgecolors='black', linewidth=0.5)
ax1.axhline(y=0, color='red', linestyle='--', linewidth=2)
ax1.axhline(y=train_residuals.std()*2, color='orange', linestyle=':', linewidth=1.5, label='+2σ')
ax1.axhline(y=-train_residuals.std()*2, color='orange', linestyle=':', linewidth=1.5, label='-2σ')
ax1.set_xlabel('Predicted HTC (MPa)', fontsize=11, fontweight='bold')
ax1.set_ylabel('Residuals (MPa)', fontsize=11, fontweight='bold')
ax1.set_title(f'Training Set: Residuals vs Predicted\nR²={train_r2:.4f}, RMSE={train_rmse:.4f}', fontsize=12, fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# 测试集残差图
ax2 = fig.add_subplot(gs[0, 1])
ax2.scatter(test_pred, test_residuals, alpha=0.6, s=60, color='orange', edgecolors='black', linewidth=0.5)
ax2.axhline(y=0, color='red', linestyle='--', linewidth=2)
ax2.axhline(y=test_residuals.std()*2, color='blue', linestyle=':', linewidth=1.5, label='+2σ')
ax2.axhline(y=-test_residuals.std()*2, color='blue', linestyle=':', linewidth=1.5, label='-2σ')
ax2.set_xlabel('Predicted HTC (MPa)', fontsize=11, fontweight='bold')
ax2.set_ylabel('Residuals (MPa)', fontsize=11, fontweight='bold')
ax2.set_title(f'Test Set: Residuals vs Predicted\nR²={test_r2:.4f}, RMSE={test_rmse:.4f}', fontsize=12, fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

# 全数据残差图
ax3 = fig.add_subplot(gs[0, 2])
ax3.scatter(all_pred, all_residuals, alpha=0.6, s=60, color='green', edgecolors='black', linewidth=0.5)
ax3.axhline(y=0, color='red', linestyle='--', linewidth=2)
ax3.axhline(y=all_residuals.std()*2, color='purple', linestyle=':', linewidth=1.5, label='+2σ')
ax3.axhline(y=-all_residuals.std()*2, color='purple', linestyle=':', linewidth=1.5, label='-2σ')
ax3.set_xlabel('Predicted HTC (MPa)', fontsize=11, fontweight='bold')
ax3.set_ylabel('Residuals (MPa)', fontsize=11, fontweight='bold')
ax3.set_title('All Data: Residuals vs Predicted', fontsize=12, fontweight='bold')
ax3.legend(fontsize=9)
ax3.grid(True, alpha=0.3)

# 残差分布
ax4 = fig.add_subplot(gs[1, 0])
ax4.hist(all_residuals, bins=15, density=True, alpha=0.7, color='skyblue', edgecolor='black', label='Residuals')
x = np.linspace(all_residuals.min(), all_residuals.max(), 100)
ax4.plot(x, stats.norm.pdf(x, mu_all, sigma_all), 'r-', linewidth=2, label='Normal Distribution')
ax4.set_xlabel('Residuals (MPa)', fontsize=11, fontweight='bold')
ax4.set_ylabel('Density', fontsize=11, fontweight='bold')
ax4.set_title(f'Residuals Distribution\nμ={mu_all:.4f}, σ={sigma_all:.4f}', fontsize=12, fontweight='bold')
ax4.legend(fontsize=9)
ax4.grid(True, alpha=0.3)

# Q-Q图
ax5 = fig.add_subplot(gs[1, 1])
stats.probplot(all_residuals_std, dist="norm", plot=ax5)
ax5.set_xlabel('Theoretical Quantiles', fontsize=11, fontweight='bold')
ax5.set_ylabel('Sample Quantiles', fontsize=11, fontweight='bold')
ax5.set_title('Q-Q Plot (Normality Test)', fontsize=12, fontweight='bold')
ax5.grid(True, alpha=0.3)

# 箱线图
ax6 = fig.add_subplot(gs[1, 2])
box_data = [train_residuals, test_residuals, all_residuals]
bp = ax6.boxplot(box_data, labels=['Train', 'Test', 'All'], patch_artist=True, showmeans=True)
colors = ['lightblue', 'orange', 'lightgreen']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
ax6.axhline(y=0, color='red', linestyle='--', linewidth=2)
ax6.set_ylabel('Residuals (MPa)', fontsize=11, fontweight='bold')
ax6.set_title('Residuals Distribution Comparison', fontsize=12, fontweight='bold')
ax6.grid(True, alpha=0.3, axis='y')

# 标准化残差
ax7 = fig.add_subplot(gs[2, 0])
ax7.scatter(range(len(all_residuals_std)), all_residuals_std, alpha=0.6, s=40, edgecolors='black', linewidth=0.5)
ax7.axhline(y=0, color='red', linestyle='--', linewidth=2)
ax7.axhline(y=2, color='orange', linestyle=':', linewidth=1.5, label='±2σ')
ax7.axhline(y=-2, color='orange', linestyle=':', linewidth=1.5)
ax7.axhline(y=3, color='red', linestyle=':', linewidth=1.5, label='±3σ')
ax7.axhline(y=-3, color='red', linestyle=':', linewidth=1.5)
ax7.set_xlabel('Sample Index', fontsize=11, fontweight='bold')
ax7.set_ylabel('Standardized Residuals', fontsize=11, fontweight='bold')
ax7.set_title('Standardized Residuals vs Index', fontsize=12, fontweight='bold')
ax7.legend(fontsize=9)
ax7.grid(True, alpha=0.3)

# 异方差性检查
ax8 = fig.add_subplot(gs[2, 1])
ax8.scatter(all_pred, np.abs(all_residuals), alpha=0.6, s=60, color='purple', edgecolors='black', linewidth=0.5)
ax8.plot(trend_line_x, trend_line_y, "r--", linewidth=2, label='Trend Line')
ax8.set_xlabel('Predicted HTC (MPa)', fontsize=11, fontweight='bold')
ax8.set_ylabel('|Residuals| (MPa)', fontsize=11, fontweight='bold')
ax8.set_title('Heteroscedasticity Check', fontsize=12, fontweight='bold')
ax8.legend(fontsize=9)
ax8.grid(True, alpha=0.3)

# 统计检验结果
ax9 = fig.add_subplot(gs[2, 2])
ax9.axis('off')
test_results = f"""
Statistical Test Results:

1. Normality Tests:
   • Shapiro-Wilk: p={shapiro_p:.4f}
   • K-S Test: p={ks_p:.4f}
   • Jarque-Bera: p={jb_p:.4f}

2. Independence Test:
   • Durbin-Watson: {dw_stat:.4f}

3. Homoscedasticity:
   • Breusch-Pagan: p={bp_p:.4f}

4. Correlation:
   • Pred vs Residual: r={corr_all:.4f}

Model Performance:
   • Train R²: {train_r2:.4f}
   • Test R²: {test_r2:.4f}
   • Train RMSE: {train_rmse:.4f}
   • Test RMSE: {test_rmse:.4f}

Passed: {passed_tests:.0f}/7 ({passed_tests/7*100:.0f}%)
"""
ax9.text(0.1, 0.5, test_results, fontsize=9, family='monospace',
         verticalalignment='center', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('Comprehensive Residual Analysis (Model Inference Only)', fontsize=16, fontweight='bold', y=0.995)
plt.savefig(os.path.join(output_dir, '1_comprehensive_residual_analysis.png'), dpi=300, bbox_inches='tight')
plt.close()

# 图2: 正态性检验详细图（新增）
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 训练集Q-Q图
ax = axes[0, 0]
stats.probplot(train_residuals_std, dist="norm", plot=ax)
ax.set_title(f'Training Set Q-Q Plot\nShapiro-Wilk p={shapiro(train_residuals)[1]:.4f}', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)

# 测试集Q-Q图
ax = axes[0, 1]
stats.probplot(test_residuals_std, dist="norm", plot=ax)
ax.set_title(f'Test Set Q-Q Plot\nShapiro-Wilk p={shapiro(test_residuals)[1]:.4f}', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)

# 训练集残差分布
ax = axes[1, 0]
ax.hist(train_residuals, bins=12, density=True, alpha=0.7, color='blue', edgecolor='black', label='Train Residuals')
kde = gaussian_kde(train_residuals)
x_range = np.linspace(train_residuals.min(), train_residuals.max(), 100)
ax.plot(x_range, kde(x_range), 'r-', linewidth=2, label='KDE')
mu, sigma = train_residuals.mean(), train_residuals.std()
ax.plot(x_range, stats.norm.pdf(x_range, mu, sigma), 'g--', linewidth=2, label='Normal Fit')
ax.set_xlabel('Residuals (MPa)', fontsize=11, fontweight='bold')
ax.set_ylabel('Density', fontsize=11, fontweight='bold')
ax.set_title('Training Set Residuals Distribution', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# 测试集残差分布
ax = axes[1, 1]
ax.hist(test_residuals, bins=8, density=True, alpha=0.7, color='orange', edgecolor='black', label='Test Residuals')
kde_test = gaussian_kde(test_residuals)
x_range_test = np.linspace(test_residuals.min(), test_residuals.max(), 100)
ax.plot(x_range_test, kde_test(x_range_test), 'r-', linewidth=2, label='KDE')
mu_test, sigma_test = test_residuals.mean(), test_residuals.std()
ax.plot(x_range_test, stats.norm.pdf(x_range_test, mu_test, sigma_test), 'g--', linewidth=2, label='Normal Fit')
ax.set_xlabel('Residuals (MPa)', fontsize=11, fontweight='bold')
ax.set_ylabel('Density', fontsize=11, fontweight='bold')
ax.set_title('Test Set Residuals Distribution', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(output_dir, '2_normality_tests_detailed.png'), dpi=300, bbox_inches='tight')
plt.close()

# 图3: 独立性和随机性检验图（新增）
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# ACF图
ax = axes[0, 0]
if max_lags_acf > 0:
    plot_acf(all_residuals, lags=max_lags_acf, ax=ax, alpha=0.05)
    ax.set_title('Autocorrelation Function (ACF)', fontsize=12, fontweight='bold')
else:
    ax.text(0.5, 0.5, 'Sample size too small', ha='center', va='center')
    ax.axis('off')

# PACF图
ax = axes[0, 1]
if max_lags_pacf > 0:
    try:
        plot_pacf(all_residuals, lags=max_lags_pacf, ax=ax, alpha=0.05)
        ax.set_title('Partial Autocorrelation Function (PACF)', fontsize=12, fontweight='bold')
    except ValueError:
        ax.text(0.5, 0.5, 'Sample size too small', ha='center', va='center')
        ax.axis('off')
else:
    ax.text(0.5, 0.5, 'Sample size too small', ha='center', va='center')
    ax.axis('off')

# 移动平均图
ax = axes[1, 0]
if window > 0:
    ax.plot(all_residuals, 'o-', alpha=0.5, label='Residuals', markersize=4)
    ax.plot(rolling_mean, 'r-', linewidth=2, label=f'{window}-point Moving Avg')
    ax.fill_between(range(len(all_residuals)), rolling_mean - rolling_std, rolling_mean + rolling_std, alpha=0.3, color='red')
    ax.axhline(y=0, color='black', linestyle='--', linewidth=1)
    ax.set_xlabel('Sample Index', fontsize=11, fontweight='bold')
    ax.set_ylabel('Residuals (MPa)', fontsize=11, fontweight='bold')
    ax.set_title('Residuals with Moving Average', fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
else:
    ax.text(0.5, 0.5, 'Sample size too small', ha='center', va='center')
    ax.axis('off')

# CUSUM图
ax = axes[1, 1]
ax.plot(cumsum_residuals, 'b-', linewidth=2)
ax.axhline(y=0, color='red', linestyle='--', linewidth=2)
ax.set_xlabel('Sample Index', fontsize=11, fontweight='bold')
ax.set_ylabel('Cumulative Sum of Residuals', fontsize=11, fontweight='bold')
ax.set_title('CUSUM Chart (Systematic Bias Check)', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(output_dir, '3_independence_and_randomness.png'), dpi=300, bbox_inches='tight')
plt.close()

print(f"  ✓ 所有可视化图片已保存")

# ====================== 生成文字报告 ======================
report_path = os.path.join(output_dir, 'Complete_Residual_Analysis_Report.txt')
with open(report_path, 'w', encoding='utf-8') as f:
    f.write("="*80 + "\n完整残差分析报告（模型推理版）\nComplete Residual Analysis Report (Inference Only)\n" + "="*80 + "\n\n")
    f.write("1. 模型信息\n" + "-"*80 + "\n")
    f.write(f"模型文件: {model_path}\n")
    f.write(f"随机种子: {saved_seed}\n")
    f.write(f"模型架构: 6→32→16→1 (865 parameters)\n")
    f.write(f"特征: {', '.join(final_features_rfe)}\n\n")
    
    f.write("2. 数据概况\n" + "-"*80 + "\n")
    f.write(f"总样本数: {len(y_all_np)} | 训练集: {len(y_train_np)} | 测试集: {len(y_test_np)}\n")
    f.write(f"测试集比例: 20%\n\n")
    
    f.write("3. 模型性能\n" + "-"*80 + "\n")
    f.write(f"训练集: R²={train_r2:.4f}, RMSE={train_rmse:.4f}, MAE={train_mae:.4f}, MAPE={train_mape:.2f}%\n")
    f.write(f"测试集: R²={test_r2:.4f}, RMSE={test_rmse:.4f}, MAE={test_mae:.4f}, MAPE={test_mape:.2f}%\n\n")
    
    f.write("4. 残差统计\n" + "-"*80 + "\n")
    f.write(f"训练集: Mean={train_residuals.mean():.4f}, Std={train_residuals.std():.4f}\n")
    f.write(f"测试集: Mean={test_residuals.mean():.4f}, Std={test_residuals.std():.4f}\n")
    f.write(f"全数据: Mean={all_residuals.mean():.4f}, Std={all_residuals.std():.4f}\n\n")
    
    f.write("5. 统计检验结果\n" + "-"*80 + "\n")
    f.write(f"正态性检验:\n")
    f.write(f"  • Shapiro-Wilk: p={shapiro_p:.4f} {'✓ Pass' if shapiro_p > 0.05 else '⚠ Attention'}\n")
    f.write(f"  • K-S Test: p={ks_p:.4f} {'✓ Pass' if ks_p > 0.05 else '⚠ Attention'}\n")
    f.write(f"  • Jarque-Bera: p={jb_p:.4f} {'✓ Pass' if jb_p > 0.05 else '⚠ Attention'}\n")
    f.write(f"  • Anderson-Darling: {anderson_result.statistic:.4f} {'✓ Pass' if anderson_result.statistic < anderson_result.critical_values[2] else '⚠ Attention'}\n\n")
    
    f.write(f"独立性检验:\n")
    f.write(f"  • Durbin-Watson: {dw_stat:.4f} {'✓ Pass' if 1.5 < dw_stat < 2.5 else '⚠ Attention'}\n\n")
    
    f.write(f"齐方差性检验:\n")
    f.write(f"  • Breusch-Pagan: p={bp_p:.4f} {'✓ Pass' if bp_p > 0.05 else '⚠ Attention'}\n\n")
    
    f.write(f"随机性检验:\n")
    f.write(f"  • Pred vs Residual Correlation: r={corr_all:.4f}, p={p_all:.4f} {'✓ Pass' if abs(corr_all) < 0.1 and p_all > 0.05 else ('✓ Weak' if abs(corr_all) < 0.3 else '⚠ Attention')}\n\n")
    
    f.write("6. 综合结论\n" + "-"*80 + "\n")
    f.write(f"通过检验: {passed_tests:.1f}/7 ({passed_tests/7*100:.1f}%)\n")
    if passed_tests >= 6:
        f.write("✓ 残差分析结果优秀，模型不存在显著系统性偏差\n")
    elif passed_tests >= 4:
        f.write("✓ 残差分析结果良好，模型基本符合统计假设\n")
    else:
        f.write("⚠ 部分统计检验未通过，建议结合可视化结果综合判断\n")
    
    f.write("\n7. 输出文件\n" + "-"*80 + "\n")
    f.write(f"Excel数据文件: Complete_Visualization_Data_For_Origin.xlsx (26个数据表)\n")
    f.write(f"可视化图片:\n")
    f.write(f"  • 1_comprehensive_residual_analysis.png (综合残差分析)\n")
    f.write(f"  • 2_normality_tests_detailed.png (正态性检验详细)\n")
    f.write(f"  • 3_independence_and_randomness.png (独立性与随机性检验)\n")
    f.write(f"分析报告: Complete_Residual_Analysis_Report.txt\n")
    
    f.write("\n8. Excel数据表说明\n" + "-"*80 + "\n")
    f.write("01-03: 基础预测与残差数据\n")
    f.write("04-09: 残差分布直方图与正态拟合（训练/测试/全数据）\n")
    f.write("10-11: 核密度估计（KDE，训练/测试）\n")
    f.write("12-14: Q-Q图数据（训练/测试/全数据）\n")
    f.write("15-16: 箱线图与标准化残差\n")
    f.write("17-18: 异方差性分析\n")
    f.write("19-22: 时间序列分析（ACF, PACF, 移动平均, CUSUM）\n")
    f.write("23-26: 统计检验、描述统计、性能指标、配置信息\n")
    
    f.write("\n" + "="*80 + "\n")
    f.write(f"报告生成时间: {pd.Timestamp.now()}\n")
    f.write("="*80 + "\n")

print(f"  ✓ 文字报告已保存")

# ====================== 完成 ======================
print("\n" + "="*80)
print("✅ 完整残差分析已完成！")
print("="*80)
print(f"\n📁 所有输出文件保存在: {output_dir}")
print(f"\n📊 主要文件:")
print(f"  • Complete_Visualization_Data_For_Origin.xlsx (26个数据表，可直接用于Origin绘图)")
print(f"  • 1_comprehensive_residual_analysis.png (综合残差分析)")
print(f"  • 2_normality_tests_detailed.png (正态性检验详细)")
print(f"  • 3_independence_and_randomness.png (独立性与随机性检验)")
print(f"  • Complete_Residual_Analysis_Report.txt (详细文字报告)")
print(f"\n✨ 统计检验通过率: {passed_tests:.0f}/7 ({passed_tests/7*100:.0f}%)")
print(f"✨ 模型性能: Train R²={train_r2:.4f}, Test R²={test_r2:.4f}")
print("="*80)

模型加载与完整残差分析程序（仅推理，不训练）

[1/4] 加载数据...

[2/4] 特征工程...
  最终特征: ['VEC', 'am', 'Mean_热膨胀(1/k)', 'Mean_比热容J/g/k', 'Mean_Pyykko(Triple Bond) (pm)', 'Var_E13 Electron affinity']

[3/4] 加载已训练模型...
  ✓ 模型文件: D:\博一下\manuscript3-机器学习\nc\nc修改9.27\10.15\10.21\1.20-回复审稿人1-C1\HTC修复1.20-架构32_16-终极优化版1.21.8\best_model.pth
  ✓ 随机种子: 44889209
  ✓ 保存的性能: Train R²=0.9961, Test R²=0.9940
  ✓ 模型已成功加载并设置为评估模式

  使用种子 44889209 划分数据...

  进行模型推理...

  验证模型性能:
  训练集: R²=0.9974, RMSE=3.4858, MAE=2.7649
  测试集: R²=0.9993, RMSE=3.1902, MAE=2.6813

[4/4] 完整残差分析...

  残差统计:
  训练集: Mean=1.5181, Std=3.1378
  测试集: Mean=1.8535, Std=2.5964
  全数据: Mean=1.5871, Std=3.0373

  统计检验:
  Shapiro-Wilk: p=0.1884 ✓
  K-S Test: p=0.8007 ✓
  Jarque-Bera: p=0.1646 ✓
  Durbin-Watson: 1.8305 ✓
  Breusch-Pagan: p=0.7927 ✓
  Pred vs Residual: r=-0.0534 ✓
  通过检验: 7.0/7 (100.0%)

  生成所有可视化数据...

  保存所有数据到Excel...
  ✓ Excel文件已保存: D:\博一下\manuscript3-机器学习\nc\nc修改9.27\10.15\10.21\1.20-回复审稿人1-C1\HTC修复1.20-架构32_16-终极优化版1.21.8\Complete_Residual_Ana